# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [1]:
import json, sqlite3, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",140)

def find_pack_root(search_root=Path("/content")):
    for candidate in search_root.rglob("FlashEats_Classroom_Pack_V2"):
        if (candidate/"database"/"flasheats.db").exists(): return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE=find_pack_root()
if BASE is None:
    try:
        from google.colab import files
        print("Upload the FlashEats Class 7 classroom pack ZIP.")
        uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith(".zip"))
        extract_dir=Path("/content/flasheats_class7"); extract_dir.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zip_name) as z: z.extractall(extract_dir)
        BASE=find_pack_root(Path("/content"))
    except Exception as e: print(e)
if BASE is None:
    BASE=find_pack_root(Path.cwd())
if BASE is None: raise FileNotFoundError("Could not locate FlashEats_Classroom_Pack_V2")
print("Using pack:",BASE)

Using pack: /home/claude/FlashEats_Classroom_Pack_V2


In [2]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])
restaurant_status=pd.read_csv(BASE/"data"/"restaurant_status.csv")
print("restaurant_status",restaurant_status.shape)

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate
restaurant_status (502, 4)


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

In [3]:
base_orders=orders.drop_duplicates("order_id",keep="first")
late_order=outcomes[outcomes.late_flag==1].order_id.iloc[0]
on_time_order=outcomes[outcomes.late_flag==0].order_id.iloc[0]
intervention_order=interventions.order_id.iloc[0]
print("late",late_order,"on-time",on_time_order,"intervention",intervention_order)


def build_order_timeline(order_id):
    """Collect every lifecycle signal for one order from every source, sorted by time."""
    ev = []
    o = base_orders[base_orders.order_id == order_id].iloc[0]

    ev.append((o.created_at, "ORDER_CREATED", f"customer {o.customer_id}", "order_db"))
    ev.append((o.promised_eta, "PROMISED_ETA (expectation, not an event)", "platform", "order_db"))
    if pd.notna(o.pickup_at):
        ev.append((o.pickup_at, "PICKED_UP", f"driver {o.driver_id}", "order_db"))
    if pd.notna(o.actual_delivery_at):
        ev.append((o.actual_delivery_at, "DELIVERED", f"driver {o.driver_id}", "order_db"))
    elif o.final_status.strip().lower() == "cancelled":
        ev.append((None, "CANCELLED (no timestamp stored)", "unknown", "order_db"))

    for r in restaurant_status[restaurant_status.order_id == order_id].itertuples():
        ev.append((r.last_updated_at, f"RESTAURANT_STATUS={r.status.strip()}", f"restaurant {r.restaurant_id}", "restaurant_status"))
    for r in customer_actions[customer_actions.order_id == order_id].itertuples():
        ev.append((r.action_at, r.action_type, f"customer {r.customer_id}", f"customer_app ({r.channel})"))
    for r in tickets[tickets.order_id == order_id].drop_duplicates().itertuples():
        ev.append((r.created_at, f"SUPPORT_TICKET: {r.category.strip()}", "customer", "support_platform"))
    for r in interventions[interventions.order_id == order_id].itertuples():
        ev.append((r.intervention_at, f"INTERVENTION: {r.intervention_type} ({r.reason})", r.initiated_by, "interventions"))

    tl = pd.DataFrame(ev, columns=["event_time", "event_type", "actor", "source_system"])
    tl["event_time"] = pd.to_datetime(tl["event_time"], format="mixed", errors="coerce")
    return tl.sort_values("event_time", na_position="last").reset_index(drop=True)

# Pick richer examples than "first row": a late order with customer reaction, and a
# pre-delivery intervention, so the timelines actually show the workflow.
late_with_reaction = (outcomes[outcomes.late_flag == 1]
    .merge(customer_actions[customer_actions.action_type == "SUPPORT_OPENED"][["order_id"]], on="order_id")
    .order_id.iloc[0])

iv = interventions.merge(base_orders[["order_id", "actual_delivery_at"]], on="order_id")
pre_delivery_iv = iv[(pd.to_datetime(iv.intervention_at, format="mixed") <
                      pd.to_datetime(iv.actual_delivery_at, format="mixed")) &
                     (iv.intervention_type == "RESTAURANT_CONTACT")].order_id.iloc[0]

for label, oid in [("ON TIME", on_time_order), ("LATE + customer reaction", late_with_reaction),
                   ("INTERVENTION", pre_delivery_iv)]:
    oc = outcomes.set_index("order_id").loc[oid]
    print(f"\n=== {label}: {oid} | outcome={oc.outcome_bucket} | delay_min={oc.delay_min} ===")
    display(build_order_timeline(oid))

late O00001 on-time O00003 intervention O00781

=== ON TIME: O00003 | outcome=delivered_on_time | delay_min=-4.41 ===


,event_time,event_type,actor,source_system
0,2026-08-01 19:35:00.000000,ORDER_CREATED,customer C0855,order_db
1,2026-08-01 19:48:00.000000,RESTAURANT_STATUS=preparing,restaurant R010,restaurant_status
2,2026-08-01 19:57:50.820366,PICKED_UP,driver D039,order_db
3,2026-08-01 20:13:28.061191,DELIVERED,driver D039,order_db
4,2026-08-01 20:17:52.941967,"PROMISED_ETA (expectation, not an event)",platform,order_db



=== LATE + customer reaction: O00001 | outcome=delivered_late | delay_min=10.82 ===


,event_time,event_type,actor,source_system
0,2026-08-13 12:56:00.000000,ORDER_CREATED,customer C0168,order_db
1,2026-08-13 13:08:00.000000,INTERVENTION: DRIVER_REASSIGNMENT (driver_unavailable),dispatch,interventions
2,2026-08-13 13:21:00.000000,ETA_VIEWED,customer C0168,customer_app (mobile_app)
3,2026-08-13 13:35:49.825561,PICKED_UP,driver D103,order_db
4,2026-08-13 14:11:36.000000,"PROMISED_ETA (expectation, not an event)",platform,order_db
5,2026-08-13 14:20:36.000000,SUPPORT_OPENED,customer C0168,customer_app (mobile_app)
6,2026-08-13 14:22:25.035899,DELIVERED,driver D103,order_db



=== INTERVENTION: O00681 | outcome=delivered_late | delay_min=1.02 ===


,event_time,event_type,actor,source_system
0,2026-08-21 13:08:00.000000,ORDER_CREATED,customer C0375,order_db
1,2026-08-21 13:21:00.000000,ETA_VIEWED,customer C0375,customer_app (mobile_app)
2,2026-08-21 13:21:00.000000,ETA_VIEWED,customer C0375,customer_app (mobile_app)
3,2026-08-21 13:24:00.000000,INTERVENTION: RESTAURANT_CONTACT (status_stale),support,interventions
4,2026-08-21 13:33:00.000000,ETA_VIEWED,customer C0375,customer_app (mobile_app)
5,2026-08-21 13:34:46.665969,PICKED_UP,driver D064,order_db
6,2026-08-21 14:18:38.536581,"PROMISED_ETA (expectation, not an event)",platform,order_db
7,2026-08-21 14:19:39.550357,DELIVERED,driver D064,order_db


### Challenge 1 findings

- **The timelines only line up once they're rebuilt around `order_id`.** No single source holds the journey: the order DB has 3 timestamps, and everything the customer did, what support saw and what operations tried sits in other systems.
- **The gap between order creation and pickup is a black box.** We can see creation and pickup, but not *when the food was ready* or *when the driver arrived*. Restaurant status is a single overwritten snapshot, so the timeline shows its last state, not the moment it changed.
- **The promised ETA is an expectation, not an event.** It is kept in the timeline as a marker, because "late" is defined relative to it.
- Customer behaviour has two phases. **ETA checks happen throughout the order**, even well before the promise (O00681 checks three times in the first 25 minutes). **Opening support comes after the promise is missed** (O00001 opens support 9 minutes after its promised ETA).


# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

For each table, document:
- primary key,
- important foreign keys,
- grain.

Then explain why this model is better for the project than mirroring every source-system table.

In [4]:
sources={"orders":orders,"customer_actions":customer_actions,"support_tickets":tickets,"interventions":interventions,"outcomes":outcomes}
for name,df in sources.items():
    print(name,df.shape); display(df.head(2))

tickets_u = tickets.drop_duplicates().copy()     # 1 exact duplicate ticket row

def key_report(df, pk, name, fks):
    row = {"table": name, "rows": len(df), "primary_key": pk,
           "pk_unique": df[pk].is_unique if pk in df else None, "pk_nulls": int(df[pk].isna().sum()) if pk in df else None}
    for fk, (parent, pcol) in fks.items():
        s = df[fk].dropna()
        row[f"{fk} -> {parent.__name__ if hasattr(parent,'__name__') else pcol}"] = None
        row["fk_checks"] = row.get("fk_checks", "") + f"{fk}: {100*s.isin(parent[pcol]).mean():.1f}% match, {int(df[fk].isna().sum())} null; "
    return {k: v for k, v in row.items() if v is not None}

model_check = pd.DataFrame([
    key_report(customers, "customer_id", "customers", {}),
    key_report(base_orders, "order_id", "orders", {"customer_id": (customers, "customer_id"),
                                                   "restaurant_id": (restaurants, "restaurant_id"),
                                                   "driver_id": (drivers, "driver_id")}),
    key_report(customer_actions, "action_id", "customer_actions", {"order_id": (base_orders, "order_id")}),
    key_report(tickets_u, "ticket_id", "support_tickets", {"order_id": (base_orders, "order_id")}),
    key_report(interventions, "intervention_id", "interventions", {"order_id": (base_orders, "order_id")}),
    key_report(outcomes, "order_id", "outcomes", {"order_id": (base_orders, "order_id")}),
])
display(model_check)

# Is customer_id on actions consistent with the order's customer?
m = customer_actions.merge(base_orders[["order_id", "customer_id"]], on="order_id", suffixes=("", "_order"))
print("Actions whose customer_id disagrees with the order's customer:", int((m.customer_id != m.customer_id_order).sum()))

# Cardinalities that decide how to join
print("Interventions per order:", interventions.groupby("order_id").size().value_counts().to_dict())
print("Actions per order (max):", customer_actions.groupby("order_id").size().max())

# Is the outcome table trustworthy? Recompute it from orders.
chk = base_orders.merge(outcomes, on="order_id")
rec = (pd.to_datetime(chk.actual_delivery_at, format="mixed") - pd.to_datetime(chk.promised_eta, format="mixed")).dt.total_seconds() / 60
both = rec.notna() & chk.delay_min.notna()
print(f"outcomes.delay_min matches recomputation for {int(((rec - chk.delay_min).abs() < 0.02)[both].sum())} of {int(both.sum())} classifiable orders")

# Overlapping sources NOT used as canonical
order_events = pd.read_csv(BASE / "data" / "order_events.csv")
ev_iv = order_events[order_events.source_system == "interventions"]
print("\norder_events intervention rows:", len(ev_iv), "| orders also in order_interventions:",
      f"{100 * ev_iv.order_id.isin(interventions.order_id).mean():.0f}%")
print("order_events intervention types:", sorted(ev_iv.event_type.unique()))
print("order_interventions types:     ", sorted(interventions.intervention_type.unique()))

orders (1603, 13)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


customer_actions (2365, 6)


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app


support_tickets (202, 5)


...

,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution


outcomes (1600, 6)


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
1,O00002,delivered,1,1.0,3.97,delivered_late


,table,rows,primary_key,pk_unique,pk_nulls,fk_checks
0,customers,900,customer_id,True,0,NaN
1,orders,1600,order_id,True,0,"customer_id: 100.0% match, 0 null; restaurant_id: 100.0% match, 3 null; driver_id: 100.0% match, 3 null;"
2,customer_actions,2365,action_id,True,0,"order_id: 100.0% match, 0 null;"
3,support_tickets,201,ticket_id,True,0,"order_id: 100.0% match, 3 null;"
4,interventions,430,intervention_id,True,0,"order_id: 100.0% match, 0 null;"
5,outcomes,1600,order_id,True,0,"order_id: 100.0% match, 0 null;"


Actions whose customer_id disagrees with the order's customer: 0
Interventions per order: {1: 430}
Actions per order (max): 5
outcomes.delay_min matches recomputation for 1495 of 1495 classifiable orders

order_events intervention rows: 260 | orders also in order_interventions: 29%
order_events intervention types: ['DRIVER_REASSIGNMENT', 'ETA_MESSAGE', 'PRIORITY_DISPATCH', 'REFUND_OFFER', 'RESTAURANT_CONTACT']
order_interventions types:      ['CUSTOMER_CREDIT', 'DRIVER_REASSIGNMENT', 'PRIORITY_DISPATCH', 'RESTAURANT_CONTACT']


### Canonical project model

```text
customers 1 ──< orders 1 ──< customer_actions      (app behaviour)
                  │  1 ──< support_tickets       (support interactions)
                  │  1 ──< interventions         (what ops/dispatch/support did)
                  │  1 ──1 outcomes              (what happened)
                  └── >1 restaurants, >1 drivers (dimensions)
```

| Table | Primary key | Foreign keys | Grain |
|---|---|---|---|
| `customers` | `customer_id` | — | one customer |
| `orders` | `order_id` (after de-duplicating 3 conflicting rows) | `customer_id`, `restaurant_id` (3 null), `driver_id` (3 null) | one order |
| `customer_actions` | `action_id` | `order_id`, `customer_id` (consistent with the order in 100% of rows) | one app action by a customer on an order |
| `support_tickets` | `ticket_id` (after removing 1 exact duplicate) | `order_id` (3 null) | one support ticket |
| `interventions` | `intervention_id` | `order_id` | one operational action on an order (currently at most 1 per order) |
| `outcomes` | `order_id` | `order_id` → orders (1:1) | one final outcome per order. Verified: its delay matches a recomputation from `orders` for every classifiable order |
| `order_journey` *(derived, Challenge 3)* | `order_id` | — | **one row per order**, the analysis table |

**Why not mirror every source table?** The sources are organised by *which system wrote them*, but the business question is organised by *what happened to an order*. Mirroring the sources would mean every analysis re-does the same joins and makes the same grain mistakes (joining a one-to-many actions table straight to orders inflates counts). A small order-centred model makes the grain explicit once, puts the customer, intervention and outcome side by side, and leaves out tables that don't serve the KPI.

**Canonical-source decisions:** two sources overlap with the model and are deliberately *not* canonical:
- `order_events.csv` contains 260 intervention events, but only 29% of them are on orders present in `order_interventions.csv`, and it uses different type names (`ETA_MESSAGE`, `REFUND_OFFER`).
- `customer_interactions.csv` partly duplicates tickets and app actions.

`order_interventions` and `customer_app_actions` are used as the systems of record. The disagreement is a limitation for the intervention owner to resolve. It isn't averaged away.


# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

In [5]:
actions_by_order=(customer_actions.groupby("order_id").agg(action_count=("action_id","count")).reset_index())

# 1. Customer interactions -> one row per order
act = customer_actions.assign(
    is_support=customer_actions.action_type.eq("SUPPORT_OPENED"),
    is_cancel=customer_actions.action_type.eq("CANCEL_ATTEMPTED"),
    is_eta_view=customer_actions.action_type.eq("ETA_VIEWED"))
actions_by_order = act.groupby("order_id").agg(
    action_count=("action_id", "count"),
    eta_views=("is_eta_view", "sum"),
    support_opened_app=("is_support", "any"),
    cancel_attempted=("is_cancel", "any")).reset_index()

# 2. Support tickets -> one row per order
tickets_by_order = (tickets_u.dropna(subset=["order_id"]).groupby("order_id")
                    .agg(ticket_count=("ticket_id", "nunique")).reset_index())

# 3. Interventions -> one row per order, with timing relative to delivery
iv = interventions.merge(base_orders[["order_id", "actual_delivery_at"]], on="order_id", how="left")
iv["after_delivery"] = (pd.to_datetime(iv.intervention_at, format="mixed") >
                        pd.to_datetime(iv.actual_delivery_at, format="mixed"))
interventions_by_order = iv.groupby("order_id").agg(
    intervention_count=("intervention_id", "count"),
    intervention_types=("intervention_type", lambda s: ", ".join(sorted(set(s)))),
    intervention_after_delivery=("after_delivery", "any")).reset_index()

# 4. Order-level journey: start from orders (grain = 1 per order), LEFT join aggregated tables
ts = ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]
order_journey = (base_orders[["order_id", "customer_id", "restaurant_id", "driver_id"] + ts]
    .merge(outcomes[["order_id", "final_status_norm", "outcome_bucket", "late_flag", "delay_min"]], on="order_id", how="left")
    .merge(actions_by_order, on="order_id", how="left")
    .merge(tickets_by_order, on="order_id", how="left")
    .merge(interventions_by_order, on="order_id", how="left"))

for c in ts:
    order_journey[c] = pd.to_datetime(order_journey[c], format="mixed", errors="coerce")
order_journey = order_journey.fillna({"action_count": 0, "eta_views": 0, "ticket_count": 0, "intervention_count": 0,
                                      "support_opened_app": False, "cancel_attempted": False,
                                      "intervention_after_delivery": False, "intervention_types": ""})
for c in ["support_opened_app", "cancel_attempted", "intervention_after_delivery"]:
    order_journey[c] = order_journey[c].astype(bool)
order_journey["support_opened"] = order_journey["support_opened_app"] | (order_journey["ticket_count"] > 0)
order_journey["received_intervention"] = order_journey["intervention_count"] > 0
order_journey["order_to_pickup_min"] = (order_journey.pickup_at - order_journey.created_at).dt.total_seconds() / 60
order_journey["pickup_to_delivery_min"] = (order_journey.actual_delivery_at - order_journey.pickup_at).dt.total_seconds() / 60
order_journey = order_journey.rename(columns={"final_status_norm": "final_status"})

assert order_journey["order_id"].is_unique and len(order_journey) == base_orders["order_id"].nunique(), "grain broken"

cols = ["order_id", "customer_id", "support_opened", "cancel_attempted", "intervention_count",
        "intervention_types", "final_status", "late_flag", "delay_min"]
print("order_journey:", order_journey.shape, "— one row per order (asserted)")
display(order_journey[cols].head(10))

order_journey: (1600, 24) — one row per order (asserted)


,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
0,O00001,C0168,True,False,1.0,DRIVER_REASSIGNMENT,delivered,1.0,10.82
1,O00002,C0043,False,False,0.0,,delivered,1.0,3.97
2,O00003,C0855,False,False,0.0,,delivered,0.0,-4.41
3,O00004,C0229,False,False,0.0,,cancelled,NaN,NaN
4,O00005,C0040,False,False,0.0,,delivered,1.0,21.02
5,O00006,C0152,False,False,0.0,,delivered,1.0,7.30
6,O00007,C0875,False,False,0.0,,delivered,1.0,4.27
7,O00008,C0223,True,False,1.0,CUSTOMER_CREDIT,delivered,1.0,26.86
8,O00009,C0685,False,False,0.0,,delivered,1.0,0.69
9,O00010,C0530,False,False,0.0,,delivered,0.0,-5.79


In [6]:
oj = order_journey
late = oj[oj.late_flag == 1]

print("1. Late orders with a support interaction:",
      int(late.support_opened.sum()), "of", len(late), f"({100 * late.support_opened.mean():.1f}%)")
print("2. Orders that received an intervention:", int(oj.received_intervention.sum()),
      f"({100 * oj.received_intervention.mean():.1f}% of all orders)")
print("   ...of which the intervention came AFTER delivery:", int(oj.intervention_after_delivery.sum()))
print("3. Most common intervention type:")
display(interventions.intervention_type.value_counts().rename("count").to_frame())

# Frustration: explicit signals (support, cancel attempt) or repeated ETA checking
oj["frustrated"] = oj.support_opened | oj.cancel_attempted | (oj.eta_views >= 3)
neglected = oj[oj.frustrated & (oj.late_flag == 1) & ~oj.received_intervention]
print("4. Late + frustrated journeys with NO intervention:", len(neglected),
      "of", int((oj.frustrated & (oj.late_flag == 1)).sum()), "late + frustrated journeys")
display(neglected.sort_values("delay_min", ascending=False)
        [["order_id", "support_opened", "cancel_attempted", "eta_views", "delay_min"]].head(10))

1. Late orders with a support interaction: 258 of 843 (30.6%)
2. Orders that received an intervention: 430 (26.9% of all orders)
   ...of which the intervention came AFTER delivery: 45
3. Most common intervention type:


,count
intervention_type,
DRIVER_REASSIGNMENT,155
RESTAURANT_CONTACT,116
PRIORITY_DISPATCH,95
CUSTOMER_CREDIT,64


4. Late + frustrated journeys with NO intervention: 281 of 380 late + frustrated journeys


,order_id,support_opened,cancel_attempted,eta_views,delay_min
193,O00194,True,False,0.0,38.34
459,O00460,False,False,3.0,36.88
736,O00737,True,False,2.0,36.28
302,O00303,True,False,3.0,33.16
210,O00211,True,False,3.0,32.38
1234,O01235,True,False,0.0,32.00
976,O00977,True,False,1.0,31.92
994,O00995,True,False,1.0,31.72
1196,O01197,True,False,0.0,31.61
892,O00893,True,False,3.0,30.32


### Challenge 3 findings

- **Grain protected:** every one-to-many table (actions, tickets, interventions) is aggregated to one row per order *before* joining, and the result is asserted to have exactly one row per order (1,600).
- **"Support opened"** combines the app's `SUPPORT_OPENED` action with support-platform tickets. The two systems overlap only partly (43 orders appear in both), so using either alone would under-count.
- **Answers:** 258 of 843 late orders (30.6%) had a support interaction. 430 orders (26.9%) received an intervention, 45 of them after delivery. Driver reassignment is the most common type (155). **281 of 380 late, frustrated journeys received no intervention at all.**
- **Timing matters:** intervention timing relative to delivery is kept as a column. All 45 post-delivery interventions are customer credits. They are service recovery, not attempts to prevent the delay, and treating them as treatments would bias any "does it work?" comparison.
- The **neglected-journeys** list is the most actionable output: customers visibly reacting to a late order, and nobody on the FlashEats side doing anything.


# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

For each chosen metric, document:
- metric name,
- formula,
- grain,
- why it matters,
- relationship to the project KPI.

At least one metric must represent:
- an outcome,
- a customer interaction,
- an intervention.

In [7]:
delivered = oj[oj.late_flag.notna()]
late = delivered[delivered.late_flag == 1]
pre = oj.intervention_count.gt(0) & ~oj.intervention_after_delivery

metrics = pd.DataFrame([
    ["Late Delivery Rate", "Outcome", "late orders / delivered orders with a delivery time",
     "order", round(100 * delivered.late_flag.mean(), 1), "%", "IS the project KPI"],
    ["Median order-to-pickup time (late vs on-time)", "Workflow driver",
     "median(pickup_at - created_at), split by outcome", "order",
     f"{late.order_to_pickup_min.median():.1f} vs {delivered[delivered.late_flag == 0].order_to_pickup_min.median():.1f}",
     "min", "Where delay accumulates, and the stage operations can act on (restaurant prep + driver arrival)"],
    ["Median pickup-to-delivery time (late vs on-time)", "Workflow driver",
     "median(actual_delivery_at - pickup_at), split by outcome", "order",
     f"{late.pickup_to_delivery_min.median():.1f} vs {delivered[delivered.late_flag == 0].pickup_to_delivery_min.median():.1f}",
     "min", "The transit share of delay (traffic, weather, route)"],
    ["Support contact rate on late orders", "Customer interaction",
     "late orders with support opened / late orders", "order", round(100 * late.support_opened.mean(), 1), "%",
     "Customer cost of lateness; it should fall as the KPI improves"],
    ["Pre-delivery intervention coverage of late orders", "Intervention",
     "late orders with an intervention before delivery / late orders", "order",
     round(100 * (pre & (oj.late_flag == 1)).sum() / len(late), 1), "%",
     "How often ops act before a late order lands. This is the lever the team controls"],
], columns=["metric", "type", "formula", "grain", "value", "unit", "link to KPI"])
display(metrics)

,metric,type,formula,grain,value,unit,link to KPI
0,Late Delivery Rate,Outcome,late orders / delivered orders with a delivery time,order,56.4,%,IS the project KPI
1,Median order-to-pickup time (late vs on-time),Workflow driver,"median(pickup_at - created_at), split by outcome",order,31.9 vs 19.8,min,"Where delay accumulates, and the stage operations can act on (restaurant prep + driver arrival)"
2,Median pickup-to-delivery time (late vs on-time),Workflow driver,"median(actual_delivery_at - pickup_at), split by outcome",order,48.9 vs 45.3,min,"The transit share of delay (traffic, weather, route)"
3,Support contact rate on late orders,Customer interaction,late orders with support opened / late orders,order,30.6,%,Customer cost of lateness; it should fall as the KPI improves
4,Pre-delivery intervention coverage of late orders,Intervention,late orders with an intervention before delivery / late orders,order,24.2,%,How often ops act before a late order lands. This is the lever the team controls


### Chosen metrics

| Metric | Type | Why it matters |
|---|---|---|
| **Late Delivery Rate** | Outcome | The project KPI itself (56.4%). Definition per Class 6: delivered orders only, cancelled excluded, 37 unclassifiable orders reported separately |
| **Order-to-pickup time** | Workflow driver | Shows *where* in the lifecycle delay builds up, and it's the stage operations can influence |
| **Pickup-to-delivery time** | Workflow driver | Separates transit delay (traffic, weather) from pre-pickup delay, so the two aren't blamed on each other |
| **Support contact rate on late orders** | Customer interaction | Translates lateness into customer cost and support load |
| **Pre-delivery intervention coverage** | Intervention | Measures whether ops act *before* a late order lands. Only pre-delivery interventions count, because a post-delivery credit can't prevent lateness |

Five metrics, each tied to the KPI, rather than every possible count.


# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

In [8]:
# A. Support interaction vs delay
print("A. Delay by support interaction (delivered orders)")
display(delivered.groupby("support_opened").agg(orders=("order_id", "count"),
        late_rate=("late_flag", "mean"), median_delay=("delay_min", "median")).round(3))

# B. Late rate with vs without intervention (pre-delivery only vs none; post-delivery shown separately)
delivered = delivered.assign(iv_group=pd.Series("none", index=delivered.index)
    .mask(delivered.received_intervention & ~delivered.intervention_after_delivery, "pre-delivery")
    .mask(delivered.intervention_after_delivery, "post-delivery"))
print("B. Late rate by intervention timing")
display(delivered.groupby("iv_group").agg(orders=("order_id", "count"), late_rate=("late_flag", "mean"),
        median_delay=("delay_min", "median")).round(3))

# C. Late rate by intervention type (pre-delivery only)
print("C. Late rate by intervention type (pre-delivery interventions only)")
pre_iv = delivered[delivered.iv_group == "pre-delivery"]
display(pre_iv.groupby("intervention_types").agg(orders=("order_id", "count"), late_rate=("late_flag", "mean"),
        median_delay=("delay_min", "median")).round(3).sort_values("late_rate"))
iv_reasons = interventions.merge(outcomes[["order_id", "late_flag"]], on="order_id")
print("Why interventions were triggered (selection matters):")
display(pd.crosstab(iv_reasons.intervention_type, iv_reasons.reason))

# D. Restaurants contributing the most late orders — volume AND rate
print("D. Restaurants by late-order contribution")
rest = (delivered.groupby("restaurant_id").agg(orders=("order_id", "count"), late_orders=("late_flag", "sum"),
        late_rate=("late_flag", "mean"), median_order_to_pickup=("order_to_pickup_min", "median"))
        .sort_values("late_orders", ascending=False).round(2))
display(rest.head(8))

# E. Support + intervention + still late
e = delivered[delivered.support_opened & delivered.received_intervention & (delivered.late_flag == 1)]
print(f"E. Support + intervention + still late: {len(e)} journeys")
display(e[["order_id", "intervention_types", "intervention_after_delivery", "eta_views", "delay_min"]]
        .sort_values("delay_min", ascending=False).head(10))

# Where does delay accumulate? Stage durations by outcome
print("Stage durations by outcome (median minutes)")
display(delivered.groupby("outcome_bucket")[["order_to_pickup_min", "pickup_to_delivery_min", "delay_min"]].median().round(1))

A. Delay by support interaction (delivered orders)

,orders,late_rate,median_delay
support_opened,,,
False,1203,0.486,-0.250
True,292,0.884,12.285


B. Late rate by intervention timing


,orders,late_rate,median_delay
iv_group,,,
none,1091,0.564,1.39
post-delivery,45,0.533,0.60
pre-delivery,359,0.568,1.62


C. Late rate by intervention type (pre-delivery interventions only)


...

,orders,late_orders,late_rate,median_order_to_pickup
restaurant_id,,,,
R050,35,22.0,0.63,28.19
R004,30,20.0,0.67,29.99
R024,25,20.0,0.80,29.77
R023,32,20.0,0.62,28.05
R030,38,20.0,0.53,23.43
R051,29,20.0,0.69,26.62
R045,29,19.0,0.66,30.82
R011,31,19.0,0.61,26.78


E. Support + intervention + still late: 67 journeys


,order_id,intervention_types,intervention_after_delivery,eta_views,delay_min
100,O00101,CUSTOMER_CREDIT,False,2.0,85.51
197,O00198,RESTAURANT_CONTACT,False,0.0,35.23
496,O00497,PRIORITY_DISPATCH,False,0.0,31.78
612,O00613,PRIORITY_DISPATCH,False,2.0,27.18
7,O00008,CUSTOMER_CREDIT,False,1.0,26.86
301,O00302,PRIORITY_DISPATCH,False,2.0,26.32
543,O00544,DRIVER_REASSIGNMENT,False,0.0,24.92
861,O00862,DRIVER_REASSIGNMENT,False,1.0,24.10
538,O00539,DRIVER_REASSIGNMENT,False,3.0,23.28
1577,O01578,PRIORITY_DISPATCH,False,3.0,22.76


Stage durations by outcome (median minutes)


,order_to_pickup_min,pickup_to_delivery_min,delay_min
outcome_bucket,,,
delivered_late,31.9,48.9,8.0
delivered_on_time,19.8,45.3,-5.5


### Challenge 5 findings

**A. Do orders with support interactions have higher delay?** Yes, clearly: 88.4% of orders with support contact were late (median delay +12.3 minutes), against 48.6% without (median −0.3 minutes).
*Tells the business:* support load follows lateness. *Does not prove:* that contacting support changes anything, or which way causation runs within a journey.

**B. Late rate with vs without intervention.** Orders with a pre-delivery intervention were late **56.8%** of the time, against **56.4%** with no intervention, which is no visible difference. The 45 post-delivery interventions are all customer credits, and only 53% of those orders were late, so **credits are being issued on orders the system recorded as on time**. That's either a credit-policy issue or a sign that the customer's experience of "late" differs from the timestamp.
*Tells the business:* current interventions show no visible association with better outcomes. *Does not prove:* that they don't work. Interventions are **targeted at at-risk orders** (`late_risk`, `slow_progress`, `prep_delay`), so intervened orders start out worse. Showing an effect would need a comparison group, such as randomised or rule-based assignment.

**C. Which intervention type has the lowest late rate?** Among pre-delivery interventions: priority dispatch 51.2%, restaurant contact 52.8%, driver reassignment 57.8%. The 18 pre-delivery customer credits are 100% late, which is expected, because a credit compensates for a delay rather than preventing it.
*Does not prove:* that one type is better, because each is triggered by different reasons (see the crosstab) and the groups are small.

**D. Which restaurants contribute the most late orders?** R050, R004, R024, R023, R030 and R051 each contribute about 20 late orders. Most combine high volume with slow order-to-pickup times (about 28–31 minutes median), though R030 is mainly high volume (38 orders, 53% late).
*Tells the business:* where to look first. *Does not prove:* restaurant fault. Without a driver-arrival event, a slow order-to-pickup time could be the driver arriving late.

**E. Support + intervention + still late:** 67 journeys. The top one (O00101, about 86 minutes) is one of the defective rows whose promised ETA precedes order creation (Class 6), so its delay isn't trustworthy. The rest are the clearest service failures, where the customer complained, ops acted, and the order was still late. They make good case studies for reviewing *when* interventions fire.

**Where delay accumulates:** late orders take about **12 minutes longer from order to pickup** than on-time orders, but only about **3–4 minutes longer in transit**. Most delay builds up **before pickup**, not on the road. That shifts attention away from "traffic is the problem" (Class 5) towards restaurant readiness and driver arrival.


# Challenge 6 — Connect the model to the KPI

Create a one-page project view:

`PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS`

Answer:
1. Which metrics are directly controllable by operations?
2. Which are outcomes?
3. Which missing event limits the model most?
4. What would you instrument next?

Final deliverable:
- core entities,
- key events,
- relationships,
- 3–5 metrics,
- KPI linkage,
- one modelling limitation.

### One-page project view

```text
PROJECT KPI         Reduce Late Delivery Rate (56.4% of delivered orders, any delay; 23.3% at >10 min)
    │
OUTCOME METRIC      Late Delivery Rate · median delay among late orders (~8 min)
    │
WORKFLOW / DRIVER   Order-to-pickup time  (late 31.9 vs on-time 19.8 min)  ← most delay builds up here
METRICS             Pickup-to-delivery time (late 48.9 vs on-time 45.3 min)
                    Support contact rate on late orders
    │
INTERVENTIONS       Driver reassignment · Restaurant contact · Priority dispatch   (pre-delivery, can prevent)
                    Customer credit                                               (post-delivery, recovery only)
                    → Pre-delivery intervention coverage of late orders
    │
DATA SOURCES /      orders (created, pickup, delivered, promised ETA) · order_outcomes
EVENTS              customer_app_actions (ETA views, support, cancel) · support_tickets
                    order_interventions (type, time, reason) · restaurant_status (snapshot only)
                    MISSING: driver_arrived_at_restaurant · food_ready timestamp
```

**1. Directly controllable by operations:** pre-delivery intervention coverage, order-to-pickup time (through restaurant SLAs and dispatch timing), and when interventions fire.

**2. Outcomes:** Late Delivery Rate, median delay, and support contact on late orders (a consequence of the outcome).

**3. The missing event that limits the model most:** `driver_arrived_at_restaurant`. The order-to-pickup stage is where most delay builds up, and without arrival we can't split it into restaurant preparation versus driver arrival, so we can't tell *who* should fix it.

**4. What to instrument next:**
- A driver **arrived-at-restaurant** event.
- A restaurant **food-ready** event, stored as a timestamped history instead of an overwritten status.
- The **trigger rule** for every intervention, recorded, so its effect can later be compared against a baseline.

### Final deliverable

- **Core entities:** customer, order, restaurant, driver.
- **Key events:** order created, picked up, delivered, customer app actions, support ticket, intervention, and the promised ETA as the reference point.
- **Relationships:** customer 1–N orders; order 1–N actions, tickets and interventions; order 1–1 outcome.
- **Metrics:** Late Delivery Rate, order-to-pickup time, pickup-to-delivery time, support contact rate on late orders, pre-delivery intervention coverage.
- **KPI linkage:** the order-to-pickup stage drives most of the late rate, and interventions and support are measured against it.
- **Modelling limitation:** the order-to-pickup stage can't be decomposed because there's no arrival or food-ready event. The two intervention sources also disagree (`order_events` vs `order_interventions`), so intervention analysis depends on which one is declared the system of record.


# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.